# Моделирование намерений в диалогах DialogSum-RU

Этот блокнот посвящён построению baseline-системы распознавания **намерений (intent) на уровне отдельных реплик** в русскоязычном датасете [`d0rj/dialogsum-ru`](https://huggingface.co/datasets/d0rj/dialogsum-ru).

## Контекст и предыдущие шаги

- В блокноте `07_dialogsum_ru_eda.ipynb` выполнен разведочный анализ датасета: распределения длин диалогов, частот тем (`topic`), статистика по сплитам.
- В блокноте `08_topic_modeling_dialogsum_ru.ipynb` мы построили современный пайплайн тематического моделирования (`sentence-transformers` + UMAP + HDBSCAN + локальная LLM-интерпретация Qwen) и получили **кластеризацию тем** диалогов, сохранённую в Google Drive проекта:

  - `results/tables/dialogsum_ru_topic_clusters_sota.parquet`
  - `results/tables/dialogsum_ru_topic_clusters_sota.csv`

  Каждой строке датасета сопоставлены: `cluster_id`, `cluster_name`, `cluster_description` (а также, опционально, `cluster_probability` и `outlier_score`).

## Колонки DialogSum-RU

- `id` — уникальный идентификатор диалога.
- `split` — train / validation / test.
- `dialogue` — текст диалога с маркерами говорящих `#Person1#:` и `#Person2#:`.
- `summary` — краткое содержание диалога.
- `topic` — короткая тема диалога (свободный текст на русском языке).
- `topic_clean`, `cluster_id`, `cluster_name`, `cluster_description` — результаты тематического моделирования из блокнота 08.

## Постановка задачи

**Цель.** Построить прототип системы детекции намерений **на уровне реплики** (utterance-level intent classification) и продемонстрировать, что **тематический кластер диалога** (доменный контекст из блокнота 08) можно использовать как дополнительный признак, потенциально улучшающий и обогащающий интерпретацию намерений.

**Определение намерения.** Под намерением реплики мы понимаем тип коммуникативного действия / цели говорящего в данной реплике: запрос информации, запрос услуги/действия, жалоба, договорённость, светская беседа и т. п. Намерение — это категория уровня **речевого акта**, не равная теме разговора.

## Что делает этот блокнот

1. Загружает результаты тематического моделирования из Google Drive.
2. Разбивает диалоги на реплики и формирует utterance-level датасет.
3. Размечает реплики **слабыми правилами (weak supervision)** по 6 классам намерений — это baseline-прототип, а не золотая разметка.
4. Обучает несколько baseline-моделей:
   - TF-IDF + LogisticRegression;
   - SentenceTransformer-эмбеддинги + LogisticRegression;
   - Context-aware вариант с признаками кластера темы.
5. Оценивает модели (accuracy, macro/micro/weighted F1, classification_report, confusion matrix).
6. Анализирует ошибки и обсуждает роль тематического контекста.

## Ограничения

- Метки **слабые (weak labels)** на основе ручных правил — это **не gold annotation**. Дальнейшая работа предполагает ручную разметку и более богатую схему намерений.
- Цель блокнота — baseline и интерпретация, а не SOTA: тяжёлый fine-tuning трансформеров здесь не выполняется.


## Ячейка 1 — Установка зависимостей

Раскомментируйте `pip install` при первом запуске в чистом окружении (Colab / Perplexity Compute).

In [ ]:
# cell 1: установка зависимостей
# !pip install -q pandas numpy pyarrow scikit-learn matplotlib seaborn \
#     sentence-transformers torch


## Ячейка 2 — Подключение Google Drive и пути к результатам

Подключаем Google Drive проекта и определяем пути для загрузки результатов тематического моделирования и для сохранения новых артефактов. Если Drive недоступен, мы выбрасываем понятную русскоязычную ошибку — локальный fallback отключён по требованию пользователя.

In [ ]:
# cell 2: монтирование Google Drive и пути проекта
import os

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print('Google Drive подключён.')
except ImportError:
    print('Окружение не Colab: пропускаем монтирование Google Drive.')
except Exception as e:
    print(f'Не удалось смонтировать Google Drive: {e}')

BASE_DIR = '/content/drive/MyDrive/russian-dialogue-intent-thesis'
RESULTS_DIR = f'{BASE_DIR}/results'
TABLES_DIR = f'{RESULTS_DIR}/tables'
FIGURES_DIR = f'{RESULTS_DIR}/figures'
MODELS_DIR = f'{BASE_DIR}/models'

for d in (RESULTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    try:
        os.makedirs(d, exist_ok=True)
    except OSError as e:
        print(f'Не удалось создать каталог {d}: {e}')

if not os.path.isdir(TABLES_DIR):
    raise FileNotFoundError(
        f'Каталог {TABLES_DIR} недоступен. '
        'Смонтируйте Google Drive проекта и убедитесь, что запущен блокнот 08 '
        '(он сохраняет таблицу с кластерами тем). '
        'Локальное сохранение отключено по требованию пользователя.'
    )

print(f'BASE_DIR    = {BASE_DIR}')
print(f'TABLES_DIR  = {TABLES_DIR}')
print(f'FIGURES_DIR = {FIGURES_DIR}')
print(f'MODELS_DIR  = {MODELS_DIR}')


## Ячейка 3 — Импорты и глобальные настройки

In [ ]:
# cell 3: импорты и глобальные настройки
import re
import json
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Импорты выполнены.')


## Ячейка 4 — Загрузка таблицы с кластерами тем (результат блокнота 08)

Сначала пробуем читать parquet, при отсутствии — CSV. Если оба файла недоступны, выбрасываем понятную русскоязычную ошибку с указанием, что нужно сначала запустить блокнот 08.

In [ ]:
# cell 4: загрузка результатов тематического моделирования
PARQUET_PATH = os.path.join(TABLES_DIR, 'dialogsum_ru_topic_clusters_sota.parquet')
CSV_PATH = os.path.join(TABLES_DIR, 'dialogsum_ru_topic_clusters_sota.csv')

if os.path.exists(PARQUET_PATH):
    df_clusters = pd.read_parquet(PARQUET_PATH)
    print(f'Загружен parquet: {PARQUET_PATH}')
elif os.path.exists(CSV_PATH):
    df_clusters = pd.read_csv(CSV_PATH)
    print(f'Загружен CSV: {CSV_PATH}')
else:
    raise FileNotFoundError(
        'Не найдены файлы с результатами тематического моделирования: '
        f'{PARQUET_PATH} или {CSV_PATH}. '
        'Сначала запустите блокнот 08_topic_modeling_dialogsum_ru.ipynb, '
        'который сохраняет эти артефакты в Google Drive проекта.'
    )

print(f'Размер таблицы:           {df_clusters.shape}')
print(f'Колонки:                  {list(df_clusters.columns)}')
print(f'Уникальных диалогов (id): {df_clusters["id"].nunique()}')
print(f'Распределение по split:')
print(df_clusters['split'].value_counts())

n_clusters = df_clusters.loc[df_clusters['cluster_id'] != -1, 'cluster_id'].nunique()
n_outliers = int((df_clusters['cluster_id'] == -1).sum())
print(f'Не-шумовых кластеров:     {n_clusters}')
print(f'Шумовых строк (-1):       {n_outliers}')

print('\nТоп-10 кластеров по числу диалогов:')
top_clusters = (
    df_clusters[df_clusters['cluster_id'] != -1]
    .groupby(['cluster_id', 'cluster_name'])
    .size()
    .sort_values(ascending=False)
    .head(10)
)
print(top_clusters)


## Ячейка 5 — Схема намерений

Для прототипа мы используем компактную схему из шести классов намерений, покрывающую типичные коммуникативные функции в задачно-ориентированных и социальных диалогах:

| Метка | Описание |
|---|---|
| `informational_request` | Запрос информации, фактический вопрос («что», «где», «когда», «как», «почему», «сколько», «какой» и т. п.). |
| `service_request` | Запрос услуги/действия: заказать, забронировать, починить, купить, оформить, попросить помочь. |
| `complaint` | Жалоба, выражение недовольства, описание проблемы: «не работает», «сломался», «плохо», «ошибка», «жалоба». |
| `arrangement` | Договорённости о встрече/плане/времени/месте: «давай встретимся», «завтра», «во сколько», «где». |
| `chitchat` | Светская речь: приветствия, прощания, благодарности, реплики поддержки разговора. |
| `other` | Прочее / неоднозначные случаи / реплики, не покрытые правилами. |

Схема намеренно небольшая: задача — продемонстрировать пайплайн, а не разработать полную таксономию.

In [ ]:
# cell 5: схема намерений
INTENT_LABELS = {
    'informational_request': 'Запрос информации (фактический вопрос)',
    'service_request':       'Запрос услуги или действия',
    'complaint':             'Жалоба, описание проблемы',
    'arrangement':           'Договорённость о встрече, времени, плане',
    'chitchat':              'Светская беседа: приветствия, благодарности, smalltalk',
    'other':                 'Прочее / не покрыто правилами',
}

INTENT_ORDER = list(INTENT_LABELS.keys())
print('Классы намерений:')
for k, v in INTENT_LABELS.items():
    print(f'  {k:25s} — {v}')


## Ячейка 6 — Разбор диалогов на реплики

Диалоги DialogSum-RU размечены маркерами говорящих в формате `#Person1#:` / `#Person2#:` (иногда встречаются `#Person3#:` и т. д.). Функция `parse_dialogue_utterances` разбирает строку диалога на список реплик с полями `speaker`, `utterance_text`, `turn_idx`.

Реализация устойчива к:
- разным номерам Person;
- множественным переводам строк;
- репликам без префикса (мы относим их к одной «неизвестной» реплике).

In [ ]:
# cell 6: парсинг реплик диалога
SPEAKER_RE = re.compile(r'#(Person\d+)#\s*:\s*', re.IGNORECASE)

def parse_dialogue_utterances(dialogue):
    """Парсит строку диалога DialogSum в список реплик.

    Возвращает список словарей: {speaker, utterance_text, turn_idx}.
    """
    if not isinstance(dialogue, str) or not dialogue.strip():
        return []

    text = dialogue.replace('\r\n', '\n').replace('\r', '\n')
    matches = list(SPEAKER_RE.finditer(text))
    if not matches:
        cleaned = text.strip()
        return [{'speaker': 'Unknown', 'utterance_text': cleaned, 'turn_idx': 0}] if cleaned else []

    utterances = []
    for i, m in enumerate(matches):
        speaker = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        utt = text[start:end].strip()
        utt = re.sub(r'\s+', ' ', utt).strip()
        if utt:
            utterances.append({
                'speaker': speaker,
                'utterance_text': utt,
                'turn_idx': i,
            })
    return utterances


sample_dlg = df_clusters['dialogue'].dropna().iloc[0]
sample_utts = parse_dialogue_utterances(sample_dlg)
print(f'Пример: разобрано {len(sample_utts)} реплик.')
for u in sample_utts[:4]:
    print(f"  [{u['turn_idx']}] {u['speaker']}: {u['utterance_text'][:120]}")


## Ячейка 7 — Построение utterance-level датасета

Из каждого диалога извлекаем реплики и переносим контекст: `cluster_id`, `cluster_name`, `topic`, `summary`, `split`.

Параметры:

- `USE_ONLY_PERSON1` — флаг, оставлять ли только реплики `Person1` (пользователь-подобный говорящий) или все реплики.
- `MIN_UTT_LEN` — отбрасываем слишком короткие реплики (по числу символов).
- `TOP_K_CLUSTERS` — сколько крупнейших не-шумовых кластеров оставить для демонстрации.
- `MAX_DIALOGUES_PER_CLUSTER` — верхняя граница диалогов на кластер (для балансировки и скорости).

In [ ]:
# cell 7: utterance-level DataFrame
USE_ONLY_PERSON1 = True
MIN_UTT_LEN = 3
TOP_K_CLUSTERS = 10
MAX_DIALOGUES_PER_CLUSTER = 100

df_non_noise = df_clusters[df_clusters['cluster_id'] != -1].copy()
cluster_sizes = (
    df_non_noise.groupby('cluster_id').size().sort_values(ascending=False)
)
top_cluster_ids = cluster_sizes.head(TOP_K_CLUSTERS).index.tolist()
print(f'Отобрано кластеров: {len(top_cluster_ids)} | id: {top_cluster_ids}')

df_sel = df_non_noise[df_non_noise['cluster_id'].isin(top_cluster_ids)].copy()

df_sel = (
    df_sel.groupby('cluster_id', group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), MAX_DIALOGUES_PER_CLUSTER), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)
print(f'Диалогов после отбора: {len(df_sel)}')

records = []
for _, row in df_sel.iterrows():
    utts = parse_dialogue_utterances(row['dialogue'])
    for u in utts:
        if USE_ONLY_PERSON1 and u['speaker'].lower() != 'person1':
            continue
        if len(u['utterance_text']) < MIN_UTT_LEN:
            continue
        records.append({
            'utterance_id': f"{row['id']}__{u['turn_idx']}",
            'dialogue_id': row['id'],
            'split': row['split'],
            'cluster_id': int(row['cluster_id']),
            'cluster_name': row['cluster_name'],
            'speaker': u['speaker'],
            'turn_idx': u['turn_idx'],
            'utterance_text': u['utterance_text'],
            'topic': row.get('topic', ''),
            'summary': row.get('summary', ''),
        })

df_utt = pd.DataFrame(records)
print(f'Реплик в utterance-level датасете: {len(df_utt)}')
print(f'Уникальных диалогов:                {df_utt["dialogue_id"].nunique()}')
print('Распределение реплик по кластерам:')
print(df_utt['cluster_name'].value_counts().head(15))
df_utt.head(5)


## Ячейка 8 — Слабая разметка намерений по правилам (weak supervision)

> **Важно.** Ниже мы строим **прототипную** разметку на основе ручных регулярных выражений и триггер-слов. Это **не gold annotation**. Цель — получить разумные слабые метки для baseline-классификаторов и продемонстрировать пайплайн.

Стратегия разметки:

1. Проверяем триггеры жалобы (`complaint`) — обычно наиболее специфичны.
2. Затем проверяем `service_request` — глаголы заказа / просьбы действий.
3. Затем `arrangement` — слова о времени, месте, договорённостях.
4. Затем `informational_request` — наличие вопросительного знака и/или вопросительных слов.
5. Затем `chitchat` — приветствия/благодарности/прощания.
6. В остальных случаях — `other`.

В `intent_rule_reason` сохраняем, какое именно правило сработало.

In [ ]:
# cell 8: правило-ориентированная разметка намерений
COMPLAINT_PAT = re.compile(
    r'\b('
    r'жалоб\w*|недовол\w*|плох(?:о|ой|ие)|ужасн\w*|отвратительн\w*|'
    r'не\s+работает|не\s+работают|сломал\w*|сломан\w*|поломк\w*|'
    r'проблем\w*|ошибк\w*|глюч\w*|зависа\w*|тормоз\w*|'
    r'возмущ\w*|раздража\w*|разочарова\w*'
    r')\b', re.IGNORECASE
)

SERVICE_PAT = re.compile(
    r'\b('
    r'заказа\w*|закаж\w*|закажу|закажи\w*|'
    r'забронир\w*|брон[ьи]\w*|'
    r'запиш\w*|записа\w*|записать\w*|'
    r'купи\w*|куплю|приобрест\w*|оформ\w*|'
    r'почини\w*|починк\w*|отремонтир\w*|ремонт\w*|'
    r'помоги\w*|помогите|помочь|подскажи\w*|подскажите|'
    r'хочу\s+(?:заказать|купить|забронировать|записаться|оформить)|'
    r'мне\s+нужн\w*|нам\s+нужн\w*|пришлите|отправьте|'
    r'выдайте|верните|обмен\w*|возврат\w*'
    r')\b', re.IGNORECASE
)

ARRANGEMENT_PAT = re.compile(
    r'\b('
    r'давай(?:те)?|встрет\w*|встреч\w*|'
    r'завтра|сегодня|вчера|послезавтра|'
    r'утром|днём|днем|вечером|ночью|'
    r'в\s+\d{1,2}(?::\d{2})?|во\s+сколько|во-сколько|'
    r'договорим\w*|договорил\w*|договорит\w*|'
    r'план\w*|расписан\w*|график\w*|'
    r'место\s+встречи|где\s+встретим\w*'
    r')\b', re.IGNORECASE
)

INFO_WORDS_PAT = re.compile(
    r'\b('
    r'что|чего|чему|чем|'
    r'где|куда|откуда|'
    r'когда|во\s+сколько|'
    r'как|каким\s+образом|'
    r'почему|зачем|отчего|'
    r'сколько|насколько|'
    r'какой|какая|какое|какие|каков\w*|'
    r'кто|кого|кому|кем'
    r')\b', re.IGNORECASE
)

CHITCHAT_PAT = re.compile(
    r'\b('
    r'привет\w*|здравствуй\w*|здрасьте|здарова|доброе\s+утро|добрый\s+день|добрый\s+вечер|'
    r'пока|до\s+свидания|до\s+встречи|увидимся|прощай\w*|'
    r'спасибо|благодар\w*|пожалуйста|'
    r'как\s+дела|как\s+поживаешь|как\s+ты|как\s+вы|'
    r'извини\w*|прости\w*|'
    r'рад\s+(?:тебя|вас)\s+видеть'
    r')\b', re.IGNORECASE
)


def assign_intent_rule_based(text):
    # Возвращает (intent_label, reason).
    if not isinstance(text, str) or not text.strip():
        return 'other', 'empty'

    t = text.strip()

    if COMPLAINT_PAT.search(t):
        return 'complaint', 'complaint_pattern'
    if SERVICE_PAT.search(t):
        return 'service_request', 'service_pattern'
    if ARRANGEMENT_PAT.search(t):
        return 'arrangement', 'arrangement_pattern'

    has_qmark = '?' in t
    has_info_word = bool(INFO_WORDS_PAT.search(t))
    if has_qmark and has_info_word:
        return 'informational_request', 'qmark+info_word'
    if has_qmark:
        return 'informational_request', 'qmark_only'
    if has_info_word and len(t.split()) <= 12:
        return 'informational_request', 'info_word_short'

    if CHITCHAT_PAT.search(t):
        return 'chitchat', 'chitchat_pattern'

    return 'other', 'fallback'


labels_reasons = df_utt['utterance_text'].apply(assign_intent_rule_based)
df_utt['intent_label'] = [lr[0] for lr in labels_reasons]
df_utt['intent_rule_reason'] = [lr[1] for lr in labels_reasons]

print('Распределение слабых меток намерений:')
print(df_utt['intent_label'].value_counts())
print('\nПримеры по каждому классу:')
for cls in INTENT_ORDER:
    sub = df_utt[df_utt['intent_label'] == cls].head(2)
    print(f'\n--- {cls} ---')
    for _, r in sub.iterrows():
        print(f"  [{r['intent_rule_reason']}] {r['utterance_text'][:140]}")


## Ячейка 9 — Фильтрация редких классов и сохранение слабо-размеченного датасета

Если какой-то класс встречается слишком редко, baseline-классификаторам нечего учить и стратифицированный split падает. Поэтому:

- классы с числом примеров меньше `MIN_CLASS_COUNT` удаляются (с предупреждением);
- итоговый слабо-размеченный датасет сохраняется в Google Drive (CSV + parquet).

In [ ]:
# cell 9: фильтрация редких классов и сохранение в Drive
MIN_CLASS_COUNT = 20

cls_counts = df_utt['intent_label'].value_counts()
rare = cls_counts[cls_counts < MIN_CLASS_COUNT].index.tolist()
if rare:
    print(f'Предупреждение: удаляем редкие классы (<{MIN_CLASS_COUNT} примеров): {rare}')
    df_utt = df_utt[~df_utt['intent_label'].isin(rare)].reset_index(drop=True)

print(f'Размер датасета после фильтрации: {len(df_utt)}')
print(df_utt['intent_label'].value_counts())

weak_csv = os.path.join(TABLES_DIR, 'dialogsum_ru_intent_utterances_weak.csv')
weak_parquet = os.path.join(TABLES_DIR, 'dialogsum_ru_intent_utterances_weak.parquet')
df_utt.to_csv(weak_csv, index=False)
df_utt.to_parquet(weak_parquet, index=False)
print(f'Сохранено: {weak_csv}')
print(f'Сохранено: {weak_parquet}')


## Ячейка 10 — Стратифицированный train / val / test split

Делим 70 / 15 / 15 по `intent_label`. При невозможности стратификации (слишком мало примеров в каком-то классе) делаем фолбэк на обычное разбиение и выводим русскоязычное предупреждение.

In [ ]:
# cell 10: train/val/test split
y_all = df_utt['intent_label'].values
X_idx = np.arange(len(df_utt))

def _safe_split(X, y, test_size, seed):
    try:
        return train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    except ValueError as e:
        print(f'Предупреждение: стратификация невозможна ({e}). Делаем нестратифицированное разбиение.')
        return train_test_split(X, y, test_size=test_size, random_state=seed)


X_trainval_idx, X_test_idx, y_trainval, y_test = _safe_split(X_idx, y_all, test_size=0.15, seed=RANDOM_SEED)
X_train_idx, X_val_idx, y_train, y_val = _safe_split(
    X_trainval_idx, y_trainval, test_size=0.15 / 0.85, seed=RANDOM_SEED
)

train_df = df_utt.iloc[X_train_idx].reset_index(drop=True)
val_df   = df_utt.iloc[X_val_idx].reset_index(drop=True)
test_df  = df_utt.iloc[X_test_idx].reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nРаспределение классов (train):')
print(train_df['intent_label'].value_counts(normalize=True).round(3))
print('\nРаспределение классов (test):')
print(test_df['intent_label'].value_counts(normalize=True).round(3))


## Ячейка 11 — Модель A: TF-IDF + LogisticRegression

Простой и быстрый baseline. Используем `class_weight='balanced'` для борьбы с дисбалансом.

In [ ]:
# cell 11: модель A — TF-IDF + LogisticRegression
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(train_df['utterance_text'].values)
X_val_tfidf   = tfidf.transform(val_df['utterance_text'].values)
X_test_tfidf  = tfidf.transform(test_df['utterance_text'].values)

clf_tfidf = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
clf_tfidf.fit(X_train_tfidf, train_df['intent_label'].values)

pred_val_tfidf  = clf_tfidf.predict(X_val_tfidf)
pred_test_tfidf = clf_tfidf.predict(X_test_tfidf)

print('=== Модель A: TF-IDF + LogReg ===')
print(f"Val  accuracy: {accuracy_score(val_df['intent_label'], pred_val_tfidf):.4f}")
print(f"Val  macro F1: {f1_score(val_df['intent_label'], pred_val_tfidf, average='macro'):.4f}")
print(f"Test accuracy: {accuracy_score(test_df['intent_label'], pred_test_tfidf):.4f}")
print(f"Test macro F1: {f1_score(test_df['intent_label'], pred_test_tfidf, average='macro'):.4f}")
print('\nClassification report (test):')
print(classification_report(test_df['intent_label'], pred_test_tfidf, zero_division=0))


## Ячейка 12 — Модель B: SentenceTransformer + LogisticRegression

Используем многоязычный `paraphrase-multilingual-MiniLM-L12-v2` (тот же, что и в блокноте 08). Эмбеддинги — 384-мерные. GPU используется при наличии.

In [ ]:
# cell 12: модель B — SentenceTransformer + LogReg
from sentence_transformers import SentenceTransformer
import torch

EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_BATCH = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство для эмбеддингов: {device}')

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)


def embed(texts):
    return embedder.encode(
        list(texts),
        batch_size=EMB_BATCH,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


X_train_emb = embed(train_df['utterance_text'].values)
X_val_emb   = embed(val_df['utterance_text'].values)
X_test_emb  = embed(test_df['utterance_text'].values)

clf_emb = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
clf_emb.fit(X_train_emb, train_df['intent_label'].values)

pred_val_emb  = clf_emb.predict(X_val_emb)
pred_test_emb = clf_emb.predict(X_test_emb)

print('=== Модель B: SentenceTransformer + LogReg ===')
print(f"Val  accuracy: {accuracy_score(val_df['intent_label'], pred_val_emb):.4f}")
print(f"Val  macro F1: {f1_score(val_df['intent_label'], pred_val_emb, average='macro'):.4f}")
print(f"Test accuracy: {accuracy_score(test_df['intent_label'], pred_test_emb):.4f}")
print(f"Test macro F1: {f1_score(test_df['intent_label'], pred_test_emb, average='macro'):.4f}")
print('\nClassification report (test):')
print(classification_report(test_df['intent_label'], pred_test_emb, zero_division=0))


## Ячейка 13 — Модель C: Context-aware (TF-IDF + one-hot `cluster_id`)

Идея: добавить к текстовым признакам реплики **доменный контекст диалога** — тематический кластер (`cluster_id`) из блокнота 08. Кодируем `cluster_id` через `OneHotEncoder(handle_unknown='ignore')` и склеиваем с TF-IDF разреженной матрицей. Это самый простой способ проверить, помогает ли тематический контекст.

In [ ]:
# cell 13: модель C — TF-IDF + one-hot(cluster_id)
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    # старые версии sklearn используют sparse=
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)

X_train_cl = ohe.fit_transform(train_df[['cluster_id']].astype(str))
X_val_cl   = ohe.transform(val_df[['cluster_id']].astype(str))
X_test_cl  = ohe.transform(test_df[['cluster_id']].astype(str))

X_train_ctx = hstack([X_train_tfidf, X_train_cl]).tocsr()
X_val_ctx   = hstack([X_val_tfidf,   X_val_cl]).tocsr()
X_test_ctx  = hstack([X_test_tfidf,  X_test_cl]).tocsr()

clf_ctx = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
clf_ctx.fit(X_train_ctx, train_df['intent_label'].values)

pred_val_ctx  = clf_ctx.predict(X_val_ctx)
pred_test_ctx = clf_ctx.predict(X_test_ctx)

print('=== Модель C: TF-IDF + one-hot(cluster_id) + LogReg ===')
print(f"Val  accuracy: {accuracy_score(val_df['intent_label'], pred_val_ctx):.4f}")
print(f"Val  macro F1: {f1_score(val_df['intent_label'], pred_val_ctx, average='macro'):.4f}")
print(f"Test accuracy: {accuracy_score(test_df['intent_label'], pred_test_ctx):.4f}")
print(f"Test macro F1: {f1_score(test_df['intent_label'], pred_test_ctx, average='macro'):.4f}")
print('\nClassification report (test):')
print(classification_report(test_df['intent_label'], pred_test_ctx, zero_division=0))


## Ячейка 14 — Сводная таблица метрик и сохранение в Google Drive

In [ ]:
# cell 14: метрики и сохранение
def _metrics(y_true, y_pred):
    return {
        'accuracy':     accuracy_score(y_true, y_pred),
        'f1_macro':     f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'f1_micro':     f1_score(y_true, y_pred, average='micro',    zero_division=0),
        'f1_weighted':  f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


rows = []
for name, y_pred_val, y_pred_test in [
    ('tfidf_logreg',         pred_val_tfidf, pred_test_tfidf),
    ('emb_logreg',           pred_val_emb,   pred_test_emb),
    ('tfidf_cluster_logreg', pred_val_ctx,   pred_test_ctx),
]:
    m_val  = _metrics(val_df['intent_label'].values,  y_pred_val)
    m_test = _metrics(test_df['intent_label'].values, y_pred_test)
    rows.append({
        'model': name,
        **{f'val_{k}':  v for k, v in m_val.items()},
        **{f'test_{k}': v for k, v in m_test.items()},
    })

metrics_df = pd.DataFrame(rows)
print('Метрики baseline-моделей:')
print(metrics_df.round(4).to_string(index=False))

metrics_csv = os.path.join(TABLES_DIR, 'dialogsum_ru_intent_baseline_metrics.csv')
metrics_df.to_csv(metrics_csv, index=False)
print(f'\nСохранено: {metrics_csv}')

pred_df = test_df[['utterance_id', 'dialogue_id', 'split', 'cluster_id', 'cluster_name',
                   'utterance_text', 'intent_label']].copy()
pred_df = pred_df.rename(columns={'intent_label': 'true_label'})
pred_df['pred_tfidf_logreg']         = pred_test_tfidf
pred_df['pred_emb_logreg']           = pred_test_emb
pred_df['pred_tfidf_cluster_logreg'] = pred_test_ctx

pred_csv = os.path.join(TABLES_DIR, 'dialogsum_ru_intent_baseline_predictions.csv')
pred_df.to_csv(pred_csv, index=False)
print(f'Сохранено: {pred_csv}')


## Ячейка 15 — Матрицы ошибок (confusion matrices)

Строим матрицы ошибок для трёх моделей и сохраняем картинки в Google Drive.

In [ ]:
# cell 15: confusion matrices
def plot_cm(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
        cbar=False,
    )
    ax.set_xlabel('Предсказанный класс')
    ax.set_ylabel('Истинный класс (weak label)')
    ax.set_title(title)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    print(f'Сохранено: {save_path}')


labels_present = sorted(set(test_df['intent_label']) | set(pred_test_tfidf)
                        | set(pred_test_emb) | set(pred_test_ctx))

plot_cm(
    test_df['intent_label'].values, pred_test_tfidf, labels_present,
    'Матрица ошибок: TF-IDF + LogReg (test)',
    os.path.join(FIGURES_DIR, 'dialogsum_ru_intent_confusion_matrix_tfidf.png'),
)
plot_cm(
    test_df['intent_label'].values, pred_test_emb, labels_present,
    'Матрица ошибок: SentenceTransformer + LogReg (test)',
    os.path.join(FIGURES_DIR, 'dialogsum_ru_intent_confusion_matrix_emb.png'),
)
plot_cm(
    test_df['intent_label'].values, pred_test_ctx, labels_present,
    'Матрица ошибок: TF-IDF + cluster + LogReg (test)',
    os.path.join(FIGURES_DIR, 'dialogsum_ru_intent_confusion_matrix_tfidf_cluster.png'),
)


## Ячейка 16 — Анализ ошибок

Смотрим, на каких репликах baseline-модель эмбеддингов ошибается. Анализируем пересечение «истинной» (слабой) метки и предсказанной, а также **тематический кластер** и `topic`, в которых ошибка произошла — это даёт интерпретируемый разрез ошибок.

In [ ]:
# cell 16: анализ ошибок
err_df = pred_df[pred_df['true_label'] != pred_df['pred_emb_logreg']].copy()
print(f'Ошибок модели B (emb_logreg) на тесте: {len(err_df)} из {len(pred_df)}')

show_cols = ['utterance_text', 'true_label', 'pred_emb_logreg', 'cluster_name']
print('\nПримеры ошибок:')
print(err_df[show_cols].head(15).to_string(index=False))

print('\nТоп классов «истина -> предсказание»:')
err_pairs = err_df.groupby(['true_label', 'pred_emb_logreg']).size().sort_values(ascending=False)
print(err_pairs.head(15))

print('\nКластеры с наибольшим числом ошибок:')
print(err_df['cluster_name'].value_counts().head(10))


## Ячейка 17 — Обсуждение: как тематический контекст помогает интенту

- **Сужение пространства намерений.** Внутри одного тематического кластера (например, «бронирование столика», «техподдержка», «обсуждение встречи») распределение намерений сильно отличается от общего: в кластере «жалобы на технику» вероятность `complaint` априорно выше, в кластере «расписание» — `arrangement`. Это даёт сильный приор для классификатора.
- **Интерпретация.** Связка `(cluster_name, intent_label)` сразу читается человеком: «это запрос услуги в домене бронирования» — намного полезнее для аналитики, чем просто «service_request».
- **Робастность.** TF-IDF и эмбеддинги моделируют локальную форму реплики; тематический кластер моделирует **домен диалога**. Эти сигналы дополнительны: даже простая конкатенация признаков (модель C) обычно не хуже базового TF-IDF, а на «доменно-зависимых» классах (например, `service_request`) может давать прирост.
- **Перспектива.** Естественное развитие — учить совместную модель `(topic_cluster, intent)` либо использовать кластер как hard-prior через class-prior, либо обучать иерархический классификатор: сначала кластер, затем условно — намерение.

## Ячейка 18 — Итоговые выводы

1. **Контекст из блокнота 08.** В качестве доменного контекста для распознавания намерений мы переиспользовали тематические кластеры, полученные SOTA-пайплайном `embeddings + UMAP + HDBSCAN + LLM`. Это позволяет связать каждый диалог с интерпретируемым кластером тем без ручной разметки доменов.

2. **Схема намерений и слабая разметка.** Мы предложили компактную схему из шести классов намерений и реализовали прозрачные правила для **слабой разметки (weak supervision)**: набор регулярных выражений и триггер-слов на русском. Это **прототипная**, а не золотая разметка; её ограничения явно зафиксированы.

3. **Baseline-модели.** Обучены три baseline:
   - TF-IDF + LogisticRegression — быстрый и интерпретируемый минимум;
   - SentenceTransformer (`paraphrase-multilingual-MiniLM-L12-v2`) + LogReg — учитывает семантику и устойчив к перефразированию;
   - TF-IDF + one-hot(`cluster_id`) + LogReg — context-aware вариант с тематическим кластером в виде признака.

   В зависимости от состава классов и распределения данных эмбеддинговая модель может обгонять TF-IDF (особенно на семантически близких классах), а context-aware вариант — выигрывать на доменно-зависимых классах.

4. **Артефакты в Google Drive.** Все таблицы и графики сохраняются только в Google Drive проекта:
   - `results/tables/dialogsum_ru_intent_utterances_weak.{csv,parquet}` — слабо-размеченный utterance-level датасет;
   - `results/tables/dialogsum_ru_intent_baseline_metrics.csv` — сравнение моделей;
   - `results/tables/dialogsum_ru_intent_baseline_predictions.csv` — предсказания на тесте;
   - `results/figures/dialogsum_ru_intent_confusion_matrix_*.png` — матрицы ошибок.

5. **Дальнейшая работа.**
   - Заменить слабые правила на **золотую ручную разметку** (хотя бы для подмножества) и калибровать правила по ней.
   - Расширить схему намерений (добавить такие классы, как `confirmation`, `refusal`, `clarification`, `feedback`).
   - Использовать **диалоговый контекст** (предыдущие реплики), а не только текущую реплику.
   - Перейти к **совместному моделированию темы и намерения** (multi-task / иерархическая классификация) либо к fine-tuning трансформеров (`xlm-roberta`, `rubert-tiny2`) с учётом тематического кластера.
